# Analyse results from Pypsa-Eur

Sources: 
- Plot capacity - map view: https://github.com/pypsa-meets-earth/documentation/blob/main/notebooks/viz/regional_transm_system_viz.ipynb
- Analyse energy potential: https://github.com/pypsa-meets-earth/documentation/blob/main/notebooks/build_renewable_profiles.ipynb
- Analyse energy generation: https://pypsa.readthedocs.io/en/latest/examples/statistics.html

Some files are needed:
* PyPSA network file (e.g. "elec.nc" contains a lot of details and looks perfect)
* a country shape file (may be found in "resources/shapes/country_shapes.geojson")
* a renewable profile file (may be found in "resources/renewable_profiles/....nc)

## Import packages

In [ ]:
import yaml
import pypsa
import warnings
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
from datetime import datetime
from cartopy import crs as ccrs
from pypsa.plot import add_legend_circles, add_legend_lines, add_legend_patches
import os
import xarray as xr
import cartopy
import sys

PATH = "../../../pypsa-eur/"
sys.path.append(os.path.join(PATH, "scripts/"))
from _helpers import rename_techs

plt.style.use(["bmh", "matplotlibrc"])
xr.set_options(display_style="html")

## Path settings

In [ ]:
# Network file
cluster = 39
planning_year = list(range(2025, 2051, 5))
run_name = "myopic-martavp-2025-2050-5-steps-all-sectors"
#"mnt/e/H2GMA/Github/Europe/pypsa-eur/results/myopic-martavp-2025-2050-5-steps-all-sectors"

#results_path = f"{PATH}results/{run_name}/postnetworks/base_s_{cluster}_lvopt__{sector_opts}_{planning_year}.nc"
network_path = f"{PATH}resources/{run_name}/networks/base.nc"
# Country shape file
regions_onshore_path = f"{PATH}resources/{run_name}/country_shapes.geojson"
# Renewable profile file
solar_path = f"{PATH}resources/{run_name}/profile_{cluster}_solar.nc"
onwind_path = f"{PATH}resources/{run_name}/profile_{cluster}_onwind.nc" 
#ToDo: Add more REE

## Color settings

In [ ]:
def rename_techs_balances(tech):
    tech = rename_techs(tech)
    if "heat pump" in tech:
        return "ambient heat"
    elif tech in ["H2 Electrolysis"]:  # , "H2 liquefaction"]:
        return "power-to-hydrogen"
    elif "solar" in tech:
        return "solar"
    elif tech in ["Fischer-Tropsch", "methanolisation"]:
        return "power-to-liquid"
    elif tech == "DAC":
        return "direct air capture"
    elif "offshore wind" in tech:
        return "offshore wind"
    elif tech == "oil" or tech == "gas":
        return "fossil oil and gas"
    elif tech in ["BEV charger", "V2G", "Li ion", "land transport EV"]:
        return "battery electric vehicles"
    elif tech in ["biogas", "solid biomass"]:
        return "biomass"
    elif tech in ["electricity"]:
        return "residential electricity demand"
    elif tech in ["industry electricity", "agriculture electricity"]:
        return "industry electricity demand"
    elif tech in ["agriculture heat", "heat", "low-temperature heat for industry"]:
        return "heat demand"
    elif "solid biomass for industry" in tech:
        return "biomass demand"
    elif "gas for industry" in tech:
        return "methane demand"
    elif tech in ["H2 for industry", "land transport fuel cell"]:
        return "hydrogen demand"
    elif tech in [
        "kerosene for aviation",
        "naphtha for industry",
        "shipping methanol",
        "agriculture machinery oil",
    ]:
        return "liquid hydrocarbon demand"
    elif tech in [
        "transmission lines",
        "H2 pipeline",
        "H2 pipeline retrofitted",
        "H2",
        "electricity distribution grid",
        "SMR",
        "SMR CC",
        "OCGT",
        "CHP",
        "gas boiler",
        "H2 Fuel Cell",
        "resistive heater",
        "battery storage",
        "methanation",
    ]:
        return "other"
    else:
        return tech

In [ ]:
with open("/mnt/e/H2GMA/Github/Europe/analyse-h2g-a-ap3-eu/config/config.pathways-myopics.yaml") as file:
    config = yaml.safe_load(file)

In [ ]:
tech_colors = config["plotting"]["tech_colors"]

In [ ]:
# H2G-A: Changed
tech_colors["fossil oil and gas"] = "#e05b09"
tech_colors["power-to-hydrogen"] = "#e05b09"
tech_colors["direct air capture"] = "#e05b09"
tech_colors["methanolisation4"] = "#e05b09"
tech_colors["steam methane reforming CC"] = "#e05b09"
tech_colors["battery electric vehicles"] = tech_colors["BEV charger"]
tech_colors["other"] = "#454545"
tech_colors["building heat demand"] = tech_colors["heat"]
tech_colors["ambient heat"] = tech_colors["heat pump"]
tech_colors["residential electricity demand"] = "#72709c"
tech_colors["industry electricity demand"] = tech_colors["electricity"]
tech_colors["hydrogen demand"] = tech_colors["land transport fuel cell"]
tech_colors["agriculture machinery"] = tech_colors["land transport fuel cell"]
tech_colors["methane demand"] = tech_colors["methane"] #H2G-A: Double check
tech_colors["liquid hydrocarbon demand"] = tech_colors["kerosene for aviation"]
tech_colors["aviation fuels"] = tech_colors["kerosene for aviation"]
tech_colors["shipping fuels"] = tech_colors["shipping methanol"]
tech_colors["biomass demand"] = tech_colors["biogas"]
tech_colors["biogas upgrading"] = tech_colors["biogas"]
tech_colors["hydrogen for industry"] = tech_colors["H2 for industry"]
tech_colors["hydrogen-to-power/heat"] = tech_colors["gas-to-power/heat"]
tech_colors["hydrogen for land transport"] = "#8487e8"

## Analyse energy system

In [ ]:
scenarios = "/mnt/e/H2GMA/Github/Europe/pypsa-eur/results/myopic-martavp-2025-2050-5-steps-all-sectors"

co2_carriers = ["co2", "co2 stored", "process emissions", "co2 sequestered"]

balances_df = pd.read_csv(
    scenarios + "/csvs/supply_energy.csv", index_col=[0, 1, 2], header=[0, 1, 2, 3]
)

balances = {i.replace(" ", "_"): [i] for i in balances_df.index.levels[0]}
balances["energy"] = [
    i for i in balances_df.index.levels[0] if i not in co2_carriers
]
balances["carbon"] = [i for i in balances_df.index.levels[0] if i in co2_carriers]

key = "energy"

df = balances_df.loc[balances[key]]

df = df.groupby(level=2).sum().div(1e6)

df.index = [
    i[:-1]
    if ((i not in ["co2", "NH3", "H2"]) and (i[-1:] in ["0", "1", "2", "3"]))
    else i
    for i in df.index
]

df = df.groupby(rename_techs_balances).sum()


#df = df.xs((onw, str(39)), level=["onw", "clusters"], axis=1)

order = pd.Index(
    [
        "fossil oil and gas",
        "hydroelectricity",
        "biomass",
        "offshore wind",
        "onshore wind",
        "solar",
        "ambient heat",
        "residential electricity demand",
        "industry electricity demand",
        "electricity demand",
        "battery electric vehicles",
        "heat demand",
        "hydrogen demand",
        "biomass demand",
        "methane demand",
        "liquid hydrocarbon demand",
        "power-to-liquid",
        "methanation",
        "power-to-hydrogen",
        "hot water storage",
        "direct air capture",
        "other",
    ]
)

order = order.intersection(df.index).append(df.index.difference(order))
df = df.loc[order]

In [ ]:
e = df

In [ ]:
# Add more than bar chart
fig, ax1 = plt.subplots(1, 1, figsize=(6.5, 5.2))  # Adjust the figure size as needed

lim = 15000
pd.DataFrame(e[('39', 'vopt', '144H-T-H-B-I-A-solar+p3-linemaxext10-onwind+p0', '2050')]).T.plot.bar(ax=ax1, width=0.8, stacked=True, color=tech_colors, ylim=(-lim, lim))

ax1.legend(loc=(1.03, -0.5), ncol=1)

# Rotate x-axis labels and format the plot
for tick in ax1.get_xticklabels():
    tick.set_rotation(90)
ax1.axhline(0, color="k", linewidth=0.75)
ax1.xaxis.set_tick_params(pad=5)
ax1.set_xlabel("network expansion", fontsize=10)
ax1.grid(axis="y")

# Set labels and title
ax1.set_ylabel(r"demand $\leftarrow$ TWh $\rightarrow$ supply   ")
ax1.set_title("Energy\nBalance", fontsize=11)

# Adjust layout and save the plot
plt.tight_layout()
#plt.savefig(OUTPUT + "energy_balance.pdf", bbox_inches="tight")

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(6.5, 5.2))

lim = 20000

# Define the years for the x-axis (2025, 2030, 2035, 2040, 2045, 2050)
years = ['2025', '2030', '2035', '2040', '2045', '2050']

# Extract all the data at once
data_dict_all = pd.DataFrame()

for year in years:
    data_dict = pd.DataFrame(e[('39', 'vopt', '144H-T-H-B-I-A-solar+p3-linemaxext10-onwind+p0', year)])
    data_dict.columns = [year]
    data_dict_all = pd.concat([data_dict_all, data_dict], axis=1) #, ignore_index=True)

# Plot all data at once
data_dict_all.T.plot.bar(ax=ax1, width=0.8, stacked=True, color=tech_colors, ylim=(-lim, lim))

ax1.legend(loc=(1.03, 0), ncol=2)

for ax in [ax1]:
    for tick in ax.get_xticklabels():
        tick.set_rotation(0)
    ax.axhline(0, color="k", linewidth=0.75)
    ax.xaxis.set_tick_params(pad=5)
    #ax.set_xlabel("network expansion", fontsize=10)
    ax.grid(axis="y")

ax1.set_ylabel(r"demand $\leftarrow$ TWh $\rightarrow$ supply   ")

ax1.set_title("Energy\nBalance", fontsize=11)

plt.tight_layout()

In [ ]:
n = pypsa.Network("/mnt/e/H2GMA/Github/Europe/pypsa-eur/results/myopic-martavp-2025-2050-5-steps-all-sectors/postnetworks/base_s_39_lvopt__144H-T-H-B-I-A-solar+p3-dist1_2050.nc")